# DSAN 6000 Homework 7: A Single Spark Cluster Can Light a Prairie Fire

As a reminder: the **code** within this notebook is provided for you, since the goal for this one is *just* to get a Spark cluster up and running, *not* to obtain proficiency with PySpark coding (that will come in the next HW!).

So, **your** main job is to **carry out the steps in the** [**Week 8 slides checklist**](https://jjacobs.me/dsan6000/w08/slides.html#/in-class-demo-2-your-first-spark-cluster) using (mainly) your **AWS Console**. Then, **once they have been run**, you will know that you've set up your cluster correctly if the Python code cells below "just work" as provided – if they don't, we have provided some hints in the instructions below, with some common error messages and what they mean. If you're still having trouble getting the code to run, feel free to post on the Google Space or [email us](mailto:dsan6000@georgetown.edu)!

## Setup Part 1: Launching the Master Node Process on *This* EC2 Instance

As shown in the checklist linked above, once you have **updated the Security Group settings** to allow incoming connections from instances within the Security Group, the next steps are to (a) install OpenJDK 21 and then (b) download the latest version of Spark using `wget`, into your **home** directory (the directory at path `~/`) on this EC2 instance, and (c) extract the `.tgz` file using `tar -xzvf <filename>.tgz`.

Once the Spark `.tgz` file has been extracted, you are set up to open the **Integrated Terminal** here in VSCode and navigate (using `cd`) into the `sbin` directory, where you should see the `start-master.sh` file – this is the key step!

However, you *shouldn not* run this `.sh` file without any arguments, as the default settings won't allow worker nodes to connect to the master process. Instead, you need to **find the *private* IP address** for this EC2 instance, either by copying it from the web-based **EC2 Console**, or by executing

```bash
hostname -I
```

within the Integrated Terminal.

Once you have this **Private IP address** for the EC2 instance, you will **start the master node process** to *listen for connections* to this IP address, by running `./start-master.sh` with the `-h` argument as follows:

```bash
./start-master.sh -h <Private IP>
```

Note that this command will *not* show a status, success/failure message, or... really anything useful in its output.

To check that it ran successfully, and to monitor what is happening on the master node, you will instead use the **web-based Spark master node UI**. If your `./start-master.sh` command executed as intended, this master node monitoring UI will be **running on your EC2 instance** on **port 8080**. To actually **see** this UI from your laptop, you will need to **forward port 8080** using the **"Ports"** tab in the bottom panel of the VSCode interface (in my VSCode setup, this is the tab directly to the right of the "Terminal" tab, and before the "Jupyter" tab).

If you are not currently forwarding any ports, you should see the text "No forwarded ports. Forward a port to access your running services locally", above a "Forward a Port" button. Click that button, then just type "8080" in the Port field that appears. You should then see "8080" under "Port", followed by "localhost:8080" under "Forwarded Address".

This means that, if all has been set up correctly, you should be able to **open your local browser**, type `localhost:8080` into the address bar, and see a web UI with **"Spark Master at spark://<Private IP>:7077"** at the top (with the Spark logo in the upper left corner), as in the following screenshot:

<center>

<img src='images/spark-master.jpeg' width='50%'>

</center>

Congratulations! This means that the master node process has been successfully launched on this EC2 instance, and you are now ready to **connect to it** from within this Jupyter notebook, and control it programmatically using `PySpark`, in the next part (Code Part 1).

## Code Part 1: Initializing a `SparkSession` Connected to the Master Process

Once you have your "main" EC2 instance set up as the **master node** for your Spark cluster, the following code cell then just sets up a `SparkSession` that "plugs into" this master node, thus allowing you to **control** your cluster programmatically using `PySpark` code.

Since much of the work here is **setting the environment up correcty** to allow the functioning of the Spark cluster, lots of things can go wrong when you run the following code cell, and that's ok! A core idea here is to **learn what the error messages are telling you** in terms of what is left to set up.

To aid you in this, the website now has a full-on dedicated [**Spark debugging guide**](https://jjacobs.me/dsan6000/writeups/debugging-spark/) (that was created by... just cutting out the too-long instructions in this cell and moving them over there). For any errors you might encounter in this assignment – and errors are to be expected at this point, since Spark's setup procedure has lots of scary moving parts and obscure details tbh! – please check there first, to see if the error message you're seeing is covered.

However, if the `SparkSession` **was** set up as intended, the output from running the following cell (because the last line just displays the `spark` variable) should look something like the following:

**SparkSession - in-memory**
                
**SparkContext**

Spark UI

<dl>
  <dt>Version</dt>
    <dd><code>v4.2.0</code></dd>
  <dt>Master</dt>
    <dd><code>spark://Private IP:7077</code></dd>
  <dt>AppName</dt>
    <dd><code>UNSC_Speeches</code></dd>
</dl>

Finally, note the additional line in the `SparkSession` initialization here that you haven't seen before:

```python
.config('spark.jars.packages', 'org.apache.hadoop:hadoop-aws:3.5.0')
```

You can think of this line as the Spark "equivalent" of how you've used `import boto3` in previous assignments to interact with AWS services from within Python – it just "adds in" [additional functionality](https://mvnrepository.com/artifact/org.apache.hadoop/hadoop-aws/3.5.0) which will allow your Spark cluster to use S3 URIs like `s3a://my-s3-bucket/my-data.parquet` to work with data in **S3 buckets** the same way it works with "standard" standalone `.csv` or `.parquet` files.

In [ ]:
#| label: sparksession-stop-previous
# Check for any previously-running sessions and stop them
from pyspark.sql import SparkSession

prev_spark = SparkSession.getActiveSession()
if prev_spark is not None:
    prev_spark.stop()
else:
    print("No previously-running SparkSessions found")


In [ ]:
#| label: get-private-ip
# Get the private IP address for this instance
hostname_output = !hostname -I
master_private_ip = hostname_output[0].strip()
print(f"Master node private IP address: {master_private_ip}")

In [ ]:
#| label: build-sparksession
spark = (
    SparkSession
      .builder
      .appName('UNSC_Speeches')
      .master(f'spark://{master_private_ip}:7077')
      .config('spark.jars.packages', 'org.apache.hadoop:hadoop-aws:3.5.0')
      .getOrCreate()
)
spark

As a final "sanity check" that the `SparkSession` successfully connected to the master node process, the following code cell will print out the URI for the master node. This is a weird address, since it *looks* like a URL, but in fact it's an "internal" address that the Spark worker nodes will use to communicate to this master node:

In [ ]:
#| label: print-master-uri
print(spark.sparkContext.master)

Here it is important to keep track of the default port numbers used by Spark for different pieces of its infrastructure, so that you know what's going on at each. As you can see from the output of the previous cell, **port 7077** is reserved for worker-to-master-node messages.

More important for you is **port 4040**: Now that you've successfully created the `SparkSession` in the above code cell, you can use VSCode to forward this port to your local laptop. Then you can visit `localhost:4040` in your browser to see the **much richer `SparkSession` UI**, which shows you what's going on in much greater detail than the **master node UI at port 8080** that you used earlier. You can think of it like: the master node UI at port 8080 knows *that* something is running, but the `SparkSession` UI actually knows *what* is running – it can show you how long particular Python functions have been running, for example. Personally, I usually navigate to the **"Executors"** tab within this `SparkSession` UI to see the resources available to the cluster in detail.

Finally, when you launch the **worker nodes** in the next section, they will have their own monitoring UI, which will run on those instances at **port 8081**. We won't use this feature, however, since we're sticking to the general principle that control and monitoring of the cluster should be conducted solely through the **master node** (a principle that most use cases for Spark adhere to... Spark developers will often say e.g. that "something is wrong with your approach" if you have a need to connect directly to the worker nodes).

## Setup Part 2: Initializing and Connecting Worker Nodes

Setting up the master node is great (it's at the core of what people mean by "launching" or "spinning up" a Spark cluster), but it can't do much on its own – we now need to set up **worker nodes**, and connect them to the master process running on this EC2 instance, so that Spark knows it can **delegate** computational tasks to these nodes.

Setting up the worker nodes as you'll do in this step is also good practice for getting comfortable quickly **creating** and **interacting with** "ephemeral" EC2 instances. I say "ephemeral" here specifically in contrast to the mental model you may have used for **this** EC2 instance: so far you've been using this instance essentially like a "remote computer" that you can log into to work on DSAN 6000 homeworks. The instances you'll create to serve as **worker nodes**, however, will instead be temporary computer instances that you create *solely* to carry out some particularly computationally-intensive task. Once they have finished working on that task, you will *terminate* them!

So, for this second setup step, your job is to use the **AWS Console** to create **three additional EC2 instances**, called `worker-1`, `worker-2`, and `worker-3`, where (as shown in class) the key **new step** when creating these instances will be **ensuring that they are in the same Security Group as the master node instance**. If you set up this EC2 instance using the default settings for new EC2 instances in the AWS Console, then its security group will be called `launch-wizard-1`.

All other settings in the creation of the new instances are the same as before: Ubuntu for the OS, `t3.large` for the type, 30 GB for the root disk, and `LabInstanceProfile` for the IAM profile (within the "Advanced Settings" panel, which you have to expand, at the bottom of the Instance creation wizard).

For each of these three new instances, once AWS finishes creating them, your task will then be to **connect to** the instance and carry out the steps in the checklist. The new thing here is that **you do *not* need to launch three new VSCode sessions** to connect to these instances. Instead, you can build your familiarity with the in-browser AWS shell!

Within the **EC2 Console**, single-click on an existing instance (say, `worker-1`) within the list of instances, and you should see a "Connect" button above the list. Click that button, then confirm the default settings on the next page (there should be another orange "Connect" button in the bottom right of this screen), and you should see an **interactive terminal** that you can use to carry out the steps in the checklist. As a quick summary:

1.  Update **Ubuntu** itself using `sudo apt update`
2.  Install OpenJDK 21, using `sudo apt install`
3.  Download the latest Apache `.tgz` file using `wget`
4.  Extract the contents of the `.tgz` file using `tar -xzvf

Once these steps have been completed, you should have an **extracted folder** within the home directory of the instance. Within this extracted folder, navigate to the `sbin` subdirectory (using `cd`), where you should now see a `start-worker.sh` script. As was the case for the master node, you do *not* want to run this with the default settings. Instead, you should include a **single additional argument** after `./start-worker.sh`, that is just **the private IP address of the master node that you found above!** (Plus the port number where it is able to receive messages from worker nodes, **7077**) The command that you execute on the worker instances should thus look like:

```bash
./start-worker.sh <Master Node Private IP>:7077
```

Once you have connected **one** worker node in this way, you should **monitor** that the connections indeed worked. Keep the master node UI open at `localhost:8080` on your local laptop (the forwarded port): you will know that the connection from worker to master node has been established successfully when a new row appears in the "Worker" section, as in the following screenshot:

<center>

<img src='images/spark-firstworker.jpeg' width='50%'>

</center>

Once you have connected **all three** worker nodes to the master node, you are ready to come back here and start running the code in the following section! As a final piece of info: the following code cell will print out the number of **cores** that the master node currently detects as available for computation. So, after you've successfully run `start-worker.sh <Master Private IP>:7077` on each worker node, you should be able to come back here and re-run the following cell to see the count go up (by 2)!

In [ ]:
print(f"Available cores (minimum # partitions): {spark.sparkContext.defaultParallelism}")

## Code Part 2: "Pointing" Your `SparkSession` at the Data

Now that you've connected the worker node, and you've verified that 2 cores are **available** to the cluster, it's time to see what this "availability" of cores actually **means** in terms of how Spark operates on data. It will be easier to see what's going on "under the hood" in Spark if we now "point" this cluster at an actual **dataset**. That way, rather than just monitoring the available nodes within the cluster in the abstract using the UI, we can concretely **check how many partitions** Spark has "chopped" the full dataset into at any given point.

In other words, by indicating to Spark the dataset we'd like to work with at this point, we can see the actual **work** that Spark is doing: **monitoring the cluster** and **adaptively partitioning the data** based on the cluster's available computational capacity.

So, the following code cell is where we actually utilize the AWS extension to Spark discussed above – we use the prefix `s3a://` to tell Spark that we'd like it to scan over a file that lives in an S3 bucket and partition the nodes in the cluster to process this data in a distributed manner (as efficiently as possible).

Feel free to browse the S3 bucket contents at the given URI (meaning, the files in the `dsan6000-unsc` bucket, with prefix `unsc-speeches`). You'll see that it contains a version of the dataset mentioned in the `README.md` file, **Hive-partitioned** by year, from `year=1946` to `year=2024`.

Once you have successfully "pointed" the `SparkSession` at the dataset, the final line displays the **schema** of the `.parquet` files. You can see how, in addition to the `string` IDs for each speech (`speech_id`) and the actual full text in the `speech` column, there are numeric variables like `order` (1 for the first speech given in the session, 2 for the second, and so on), and `boolean` values like `secretary_general` (indicating whether or not the speaker is the UN Secretary General).

In [ ]:
s3_uri = "s3a://dsan6000-unsc/unsc-speeches"
unsc_sdf = (
  spark
  .read
  .option('header', 'true')
  .parquet(s3_uri)
)
unsc_sdf.printSchema()

As we discussed in class, the quickest possible way to see a result from a query (here the "query" is just, loading the dataset) is to use `take()`, since that will immediately display the first **materialized** record. So, as you can see by running the following code cell, the output of `take(1)` will be a **list** containing one element, and that element will be of type `Row`: information about the first speech in the corpus.

In [ ]:
speech_take_result = unsc_sdf.take(1)
print(speech_take_result)
print(type(speech_take_result))
first_speech = speech_take_result[0]
print(first_speech)
print(type(first_speech))

Although this `Row` type is PySpark's default for "raw" results, it can be a bit easier to *vertically* scroll through a `Row` **converted into** a Python dictionary:

In [ ]:
first_speech.asDict()

And now, on top of the *minimum* number of partitions that will be created, given the resources (cores) available to the cluster, we can also see the current *actual* number of partitions that Spark has "chopped" the data into. There is one "standard" case and two less-common cases here.

The most common case (the case described in class, to explain Spark's "goal" when partitioning the data) is to have the actual number of partitions **match** the number of available cores. However, there are two alternatives that you might see:

* Spark may create *more* partitions than there are available cores, if (for example) it experiences high latency when delivering data from the S3 bucket to the worker nodes' memories. The additional partitions help in this case since each worker node can work on a *subset* of the full (ideal) partition, while the remainder of the (ideal) partition is delivered
* Spark may also create *fewer* partitions than there are available cores: this case is slightly harder to explain (though it actually happened a few times when testing this assignment), but essentially it means Spark's Catalyst optimizer has determined that the **overhead costs** of partitioning the data into $N$ pieces, where $N$ is the number of cores available to the cluster, is too high to be worth it.

The latter explanation is *in theory* why Spark would utilize fewer partitions than available cores (and is why, in class, I emphasized that Spark's default behavior is to **aim for** a one-to-one partition-to-core mapping). In my own experience though (and in several runs of this assignment), sometimes Spark just doesn't seem to utilize all available cores even when it seems like it should be possible to do so... If I had a better explanation for this I would give it! (And, if you have ideas, dear student-reader, feel free to suggest them in class or on the Google Space 😉).

In [ ]:
print(f"Available cores (minimum # partitions): {spark.sparkContext.defaultParallelism}")
print(f"unsc_sdf # partitions: {unsc_sdf.rdd.getNumPartitions()}")

## Code Part 3: Computing Country-By-Country (Affiliation-By-Affiliation) Speech Counts

This is the part where I have **removed** from your responsibilities from this homework! The code here is written for you, and you just need to **execute** the code to produce the counts, for full points!

The following code cell uses a syntax that should **look familiar from Polars** to:

* (a) Construct an *expression* representing the computational step of grouping by affiliation (`affiliation_cow`, which is usually the *country* that the individual is speaking on behalf of) and counting, and then
* (b) Actually *execute* this expression, allowing Spark to optimize the underlying **physical plan** for carrying it out.

In case you're wondering why we use `affiliation_cow`: `affiliation` is the more unstructured description of the *entity* that the speaker is representing (as listed in the UN's official records) - this means that, e.g., 5081 speeches have `"United Kingdom"` listed as the speaker's affiliation, while 4863 of them have `"United Kingdom of Great Britain and Northern Ireland"`. `affiliation_cow` instead uses a mapping created by the University of Michigan's [Correlates of War (COW) project](https://correlatesofwar.org/), which groups these two entities together.

*(Final footnote, I promise! If you decide to use the COW mapping for your own projects, note that it groups historical entities into present-day sovereignty-holders: so, although northern and southern Yemen used to be separate countries as you saw in HW6, the COW mapping just assigns both to "Yemen". It assigns East and West Germany to "Germany", and so on... this can be bad news if your dataset stretches back to before the 1990s! You'll need to manually specify in that case, e.g., that data on East and West Germany should be summed, averaged, etc.)*

In [ ]:
#| label: country-counts
country_count_sdf = (
  unsc_sdf
  .groupBy('affiliation_cow')
  .count()
  .orderBy("count", ascending=False)
)
country_count_sdf.printSchema()

Now that we've written the expression for the computation we'd like to carry out, we can view Spark's **physical plan** for carrying out this expression on the hardware available to the cluster, using the `explain()` function:

In [ ]:
country_count_sdf.explain()

And finally, to actually **execute** this physical plan, we need to use a PySpark function that triggers **eager evaluation**: above we used `take(1)` to just get a single result, but this time we will use `toPandas()`: since we believe in advance that the results in this case *will* be able to fit into memory in full, we don't need to worry about only `take()`-ing some subset of the results, as we would for queries with many more results.

In [ ]:
#| label: execute-query
# toPandas() triggers *eager* execution of the query, *materializing* the
# results as a Pandas DataFrame, the format we know and love :)
country_count_df = country_count_sdf.toPandas()
print(f"Number of unique affiliations: {len(country_count_df)}")
display(country_count_df)
country_count_df.to_csv("affiliation_counts.csv", index=False)

And... you did it! In HW8 you will work with more intensive PySpark-specific operations (specifically, you will set up a **Machine Learning pipeline** that will also incorporate **NLP operations** on texts like these speeches!), but for now all that needs to be submitted for this homework is:

* This `.ipynb` file (with the executed cells and their outputs)
* An `affiliation_counts.csv` file, which the above code cell saves in its final line, and
* **Two screenshots** added to the **images** subdirectory:
  * One called `master-ui.jpeg` (or png) showing the **Master node UI** with three worker nodes connected to it, and
  * One called `session-ui.jpeg` (or png) showing the **Executors tab within the `SparkSession` UI**, demonstrating again that the **`SparkSession`** (and not just the master node) has three workers connected to it, with 2 cores each.

The `README.md` file in this repo also contain example screenshots, towards the end, for comparison with your own. (Since the images there have the suffix `-example.jpeg`, their filenames should not clash with yours!)

## Setup Part 3: Spark Cluster Shutdown

Once you have used your Spark cluster, via the `SparkSession` created above, to generate the `affiliation_counts.csv` file, and once you have taken the two required screenshots, you are ready to **shut down** your Spark cluster. The most straightforward way to do this is as follows:

1.  Log back into each `worker` instance (using the **Connect** button within the EC2 Console to access the shell), then navigate to the `sbin` subdirectory within the main `spark` folder and execute `./stop-worker.sh`
2.  Again for each worker, once you have run `./stop-worker.sh`, you can return to the EC2 Console and actually **Terminate** the instances (if you prefer to only **Stop** them, that's... technically ok, but it will mean greater AWS account costs, since the 30GB HDD on each worker instance requires hourly payments!)
3.  Once all worker instances have been terminated, you can now also open the **Integrated Terminal** here in VSCode, navigate back to the `sbin` subdirectory, and execute `./stop-worker.sh`
4.  Once `./stop-worker.sh` has finished, your instance is back in the state it was in before you began HW7! You can end your AWS/VSCode session as usual (as you have done for previous assignments).
